# 📄 Research Papers Recommendation System
### Using Sentence Transformers + Cosine Similarity
---

## Step 1 — Install Dependencies

In [1]:
!pip install sentence_transformers torch pandas numpy

Defaulting to user installation because normal site-packages is not writeable


## Step 2 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import pickle
import torch
import os
from sentence_transformers import SentenceTransformer, util

os.makedirs('models', exist_ok=True)
print('Libraries loaded successfully!')


Libraries loaded successfully!


In [4]:
import os
os.chdir(r"C:\Users\karatmal narendar\Desktop\MLP\research_paper_project")
print(os.getcwd())

C:\Users\karatmal narendar\Desktop\MLP\research_paper_project


## Step 3 — Load Dataset

In [5]:
arxiv_data = pd.read_csv('arxiv_data_210930-054931.csv')
print(f'Dataset shape: {arxiv_data.shape}')
print(f'Columns: {arxiv_data.columns.tolist()}')
arxiv_data.head()

Dataset shape: (56181, 3)
Columns: ['terms', 'titles', 'abstracts']


,terms,titles,abstracts
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...
3,"['cs.LG', 'cs.CR']",Releasing Graph Neural Networks with Different...,With the increasing popularity of Graph Neural...
4,['cs.LG'],Recurrence-Aware Long-Term Cognitive Network f...,Machine learning solutions for pattern classif...


## Step 4 — Clean Dataset

In [6]:
arxiv_data = arxiv_data.drop_duplicates()
arxiv_data = arxiv_data.dropna(subset=['titles', 'abstracts'])
arxiv_data = arxiv_data.reset_index(drop=True)

# Use first 5000 rows to keep it fast
arxiv_data = arxiv_data[:5000]

print(f'Clean dataset shape: {arxiv_data.shape}')
arxiv_data.head(3)

Clean dataset shape: (5000, 3)


,terms,titles,abstracts
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...


## Step 5 — Load Sentence Transformer Model

In [7]:
rec_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded!')

Model loaded!


## Step 6 — Generate Embeddings

In [8]:
sentences = arxiv_data['titles'].tolist()

print(f'Encoding {len(sentences)} paper titles...')
embeddings = rec_model.encode(sentences, show_progress_bar=True)

print(f'Embeddings shape: {embeddings.shape}')

Encoding 5000 paper titles...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


## Step 7 — Test Recommendation

In [9]:
def recommendation(input_paper, embeddings, sentences, model, k=5):
    cosine_scores = util.cos_sim(embeddings, model.encode(input_paper))
    top_similar_papers = torch.topk(cosine_scores, dim=0, k=k, sorted=True)
    papers_list = []
    for i in top_similar_papers.indices:
        papers_list.append(sentences[i.item()])
    return papers_list

# Test it
test_title = 'deep learning for natural language processing'
results = recommendation(test_title, embeddings, sentences, rec_model)

print(f'Top 5 recommendations for: "{test_title}"')
print('='*60)
for i, paper in enumerate(results, 1):
    print(f'{i}. {paper}')

Top 5 recommendations for: "deep learning for natural language processing"
1. GluonCV and GluonNLP: Deep Learning in Computer Vision and Natural Language Processing
2. Natural Language Processing (almost) from Scratch
3. NAS-Bench-NLP: Neural Architecture Search Benchmark for Natural Language Processing
4. Deep Learning for Technical Document Classification
5. Deep CNN Frame Interpolation with Lessons Learned from Natural Language Processing


## Step 8 — Save Models

In [10]:
with open('models/embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

with open('models/sentences.pkl', 'wb') as f:
    pickle.dump(sentences, f)

with open('models/rec_model.pkl', 'wb') as f:
    pickle.dump(rec_model, f)

print('All recommendation models saved to models/ folder!')
print('Now open notebook 02 to train the prediction model.')

All recommendation models saved to models/ folder!
Now open notebook 02 to train the prediction model.
